In [1]:
import os
import re
import json
import numpy as np
import cv2
import robotic as ry
from pathlib import Path
from scipy.spatial.transform import Rotation

In [2]:
PROJECT_ROOT = Path(".").resolve().parent
#PROJECT_ROOT = Path("/home/salman/rai_workspace/Voxel_manipulation_v2/pose_estimation")
SCENE_FILE   = PROJECT_ROOT / "environments/gfiles/mesh_clutter_high_30.g"
OUT_RGB      = PROJECT_ROOT / "outputs/rgb"
OUT_DEPTH    = PROJECT_ROOT / "outputs/depth"
OUT_RGB.mkdir(parents=True, exist_ok=True)
OUT_DEPTH.mkdir(parents=True, exist_ok=True)

CAM_WIDTH  = 1920
CAM_HEIGHT = 1920

# Exact camera definitions from Batch_generator.py (5 m distances, no focalLength override)
CAMERAS = [
    ("cam_dim_0", 'Q:"t(0 0 5) d(180 1 0 0)", shape:camera, size=[1], width:1920, height:1920'),
    ("cam_dim_1", 'Q:"t(0 5 1.5) d(180 1 0 0) d(-75 1 0 0) d(180 0 0 1)", shape:camera, size=[1], width:1920, height:1920'),
    ("cam_dim_2", 'Q:"t(0 -5 1.5) d(165 1 0 0) d(90 1 0 0)", shape:camera, width:1920, height:1920'),
    ("cam_dim_3", 'Q:"t(5 0 1.5) d(180 1 0 0) d(-90 1 0 0) d(195 0 0 1) d(90 0 1 0)", shape:camera, width:1920, height:1920'),
    ("cam_dim_4", 'Q:"t(-5 0 1.5) d(180 1 0 0) d(-90 1 0 0) d(165 0 0 1) d(-90 0 1 0)", shape:camera, width:1920, height:1920'),
]

In [3]:
def remove_panda(C):
    """Remove robot, target markers, and existing cameras from the scene."""
    for frame in list(C.getFrameNames()):
        if re.search(r'^l_|target|^(cameraTop|cameraWrist|panda_collCameraWrist)', frame):
            C.delFrame(frame)


def rai_pose_to_matrix(pose: np.ndarray) -> np.ndarray:
    """Convert RAI pose [x, y, z, qw, qx, qy, qz] to a 4x4 matrix."""
    pos = pose[:3]
    qw, qx, qy, qz = pose[3], pose[4], pose[5], pose[6]
    R = Rotation.from_quat([qx, qy, qz, qw]).as_matrix()
    T = np.eye(4)
    T[:3, :3] = R
    T[:3, 3]  = pos
    return T


def fxycxy_to_K(fxycxy) -> np.ndarray:
    fx, fy, cx, cy = fxycxy
    return np.array([[fx, 0, cx],
                     [0, fy, cy],
                     [0,  0,  1]], dtype=np.float64)

In [4]:
C = ry.Config()
C.addFile(str(SCENE_FILE))         # just the filename — CWD is already the gfiles dir

print("Removing robot/target frames …")
remove_panda(C)

for name, args in CAMERAS:
    C.addFrame(name=name, parent="world", args=args)

-- WARNING:stbImage.cpp:loadImage:25(-1) failed to load image file: ./RedCup_25k_tex.png
-- WARNING:i_assimp.cpp:loadMesh:213(-1) loaded texture is improper - deleting it
Removing robot/target frames …
-- WARNING:stbImage.cpp:loadImage:25(-1) failed to load image file: ./optimized_poisson_texture_mapped_mesh.png
-- WARNING:i_assimp.cpp:loadMesh:213(-1) loaded texture is improper - deleting it
-- WARNING:stbImage.cpp:loadImage:25(-1) failed to load image file: ./Sprudelflasche_25k_tex.png
-- WARNING:i_assimp.cpp:loadMesh:213(-1) loaded texture is improper - deleting it
-- WARNING:stbImage.cpp:loadImage:25(-1) failed to load image file: ./visual_hull_refined_smoothed.png
-- WARNING:i_assimp.cpp:loadMesh:213(-1) loaded texture is improper - deleting it
-- WARNING:stbImage.cpp:loadImage:25(-1) failed to load image file: ./Deodorant_25k_tex.png
-- WARNING:i_assimp.cpp:loadMesh:213(-1) loaded texture is improper - deleting it
-- WARNING:stbImage.cpp:loadImage:25(-1) failed to load image file

In [5]:
cam_view    = ry.CameraView(C)
camera_info = {}

for name, _ in CAMERAS:
    cam_view.setCamera(C.getFrame(name))
    rgb, depth = cam_view.computeImageAndDepth(C, simulateDepthNoise=False)

    K          = fxycxy_to_K(cam_view.getFxycxy())
    rgb_path   = OUT_RGB   / f"{name}.png"
    depth_path = OUT_DEPTH / f"{name}.npy"

    cv2.imwrite(str(rgb_path), cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR))
    np.save(str(depth_path), depth.astype(np.float32))

    T_world_cam = rai_pose_to_matrix(C.getFrame(name).getPose())

    camera_info[name] = {
        "K":           K.tolist(),
        "T_world_cam": T_world_cam.tolist(),
        "width":       CAM_WIDTH,
        "height":      CAM_HEIGHT,
        "rgb_path":    str(rgb_path),
        "depth_path":  str(depth_path),
    }

    dmin = depth[depth > 0].min() if (depth > 0).any() else 0.0
    dmax = depth.max()
    print(f"  {name}: rgb={rgb.shape}  depth range [{dmin:.3f}, {dmax:.3f}] m")

with open(PROJECT_ROOT / "outputs/camera_info.json", "w") as f:
    json.dump(camera_info, f, indent=2)

print(f"\nSaved RGB images  -> {OUT_RGB}")
print(f"Saved depth maps  -> {OUT_DEPTH}")
print(f"Saved camera info -> outputs/camera_info.json")

  cam_dim_0: rgb=(1920, 1920, 3)  depth range [4.220, 5.000] m
  cam_dim_1: rgb=(1920, 1920, 3)  depth range [3.270, 10.046] m
  cam_dim_2: rgb=(1920, 1920, 3)  depth range [3.270, 10.046] m
  cam_dim_3: rgb=(1920, 1920, 3)  depth range [3.270, 10.046] m
  cam_dim_4: rgb=(1920, 1920, 3)  depth range [3.270, 10.046] m

Saved RGB images  -> /home/salman/rai_workspace/Voxel_manipulation_v2/pose_estimation/outputs/rgb
Saved depth maps  -> /home/salman/rai_workspace/Voxel_manipulation_v2/pose_estimation/outputs/depth
Saved camera info -> outputs/camera_info.json
